In [8]:
from langchain.chat_models import ChatOpenAI
from langchain_ollama.llms import OllamaLLM
from langchain.chat_models import init_chat_model

## define Chat model from here https://ollama.com/library
CHAT_MODEL = 'qwen3:8b'

In [9]:
## define LLM
llm = init_chat_model(CHAT_MODEL, model_provider='ollama')



In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

#model = OllamaLLM(model=CHAT_MODEL)

chain = prompt | llm

chain.invoke({"question": "What is LangChain?"})

AIMessage(content="<think>\nOkay, the user is asking about LangChain. I need to explain what it is. Let me start by recalling what I know. LangChain is a framework for working with language models. It's designed to help developers build applications that use large language models (LLMs) like GPT-3 or others.\n\nWait, but I should break it down. First, what's the main purpose of LangChain? It allows for creating, training, and deploying models. But maybe that's too vague. Let me think. LangChain provides tools for handling the entire workflow of using LLMs. That includes things like data processing, model training, deployment, and integration with other systems.\n\nWait, maybe I should mention the components. There's the model itself, which is the core. Then there's the training data, which is the data used to train the model. Then there's the training process, which involves algorithms and optimization techniques. And deployment, where the model is integrated into an application. Oh, b

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage

## Define prompt
prompt = "Translate this sentence from English to French: I love programming."

# Invoke the chatbot using a hummanMessage to reecive an AIMessage
response = llm.invoke(
    [
        HumanMessage(
            content=prompt
        )
    ]
)

## Take the content from the output
print(response)

content='<think>\nOkay, the user wants to translate "I love programming" into French. Let me start by recalling the basic structure of this sentence. The subject is "I," which in French is "je." The verb "love" here is "aimer," but since the subject is "je," the conjugation would be "j\'aime." Then "programming" translates to "la programmation." So putting it together, it should be "J\'aime la programmation." Wait, but in French, when you have "je" as the subject, the verb comes before the subject? No, wait, in French, the subject usually comes before the verb. Wait, no, in English it\'s "I love," but in French, it\'s "Je aime" but that\'s not correct. Wait, no, the correct conjugation is "J\'aime." So the structure is "Je [verb]." So the correct translation is "J\'aime la programmation." Let me check if there\'s any other way to phrase it. Sometimes, people might say "J\'adore la programmation," but "adore" is more like "love" in a stronger sense. The original sentence uses "love," so

In [12]:

response = llm.invoke(
    [
        SystemMessage(
            content = 'You are a helpfull assistant that translates English to french'
        ),
        HumanMessage(
            content="I love programming and everything else"
        )
    ]
)

print(response)

content='<think>\nOkay, the user wants to translate "I love programming and everything else" into French. Let me start by breaking down the sentence. "I love" is straightforward; in French, that\'s "J\'aime". Now, "programming" translates to "la programmation". The next part is "and everything else". "Everything else" can be translated as "tout le reste" or "tout ce qui reste". \n\nWait, "everything else" might have a nuance. If they mean "everything else besides programming", maybe "tout ce qui reste" is better. But the original sentence is "and everything else", so it\'s adding to the previous statement. Let me check the structure. The original is "I love programming and everything else". So in French, it would be "J\'aime la programmation et tout le reste" or "J\'aime la programmation et tout ce qui reste".\n\nHmm, "tout le reste" is more like "everything else" in a general sense. "Tout ce qui reste" implies "whatever is left", which might not fit perfectly. The user might be expres

In [17]:
# def extract_thoughts(response: str) -> str:
#     """
#     Extracts the thought process from the AI's response.
#     """
#     # Extract text after </think>
#     if "</think>" in response:
#         response = response.split("</think>", 1)[1].strip()
#     else:
#         response = response

#     # Convert literal '\n' to actual newlines
#     response = response.encode('utf-8').decode('unicode_escape')

#     return response

In [18]:
# extract_thoughts(response)

### Chatbots do not interact with the real world...

Chatbots cannot interact with anything outside of their enviroment, they will only generate text from the knowledge they were trained on. Additionally, they do not have acces to external codes or applicatoin that could allow interaction with outside world.

For example, if we ask the chatbot about files in the current directory it will no be able to determine them since it cannot interact with the real world.

In [19]:
chatbot_test = llm.invoke(
    [HumanMessage(content="What are the files in the current directory?")
     ])

print(chatbot_test)

content='<think>\nOkay, the user is asking about the files in the current directory. Let me think about how to approach this.\n\nFirst, I need to figure out the current working directory. But wait, as an AI, I don\'t have access to the file system of the user\'s machine. So I can\'t actually list the files. Hmm, that\'s a problem.\n\nWait, maybe the user is using a system where they can run commands. Like in a terminal or command prompt. So perhaps the answer should guide them on how to check the files in their own directory. \n\nLet me recall the commands. In Unix-based systems (like Linux or macOS), the command is \'ls\'. In Windows, it\'s \'dir\'. But maybe the user is on a different OS. Should I mention both options?\n\nAlso, if they\'re using a graphical interface, maybe they can right-click and select "Open in terminal" or "Show contents". But I should cover the most common methods.\n\nWait, the user might not know these commands. So I should explain them clearly. Maybe start by 

### Create an agent by adding tools

In [20]:
### Reference for community tools : https://python.langchain.com/api_reference/community/tools.html
from langchain_community.tools import ShellTool
#from langchain_experimental.tools import ShellTool
## define Shell Tool
shell_tool = ShellTool()

In [21]:
## Load default OpenAI chatbot
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor,create_react_agent # To load simple ReAct agent. Reason an act
from langchain import hub

## Define LLM
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
llm = init_chat_model(CHAT_MODEL, model_provider='ollama')


# Define list of tools the LLM is going to use 
tools = [shell_tool]

# Get the template prompt to use - you can modify this!
prompt = hub.pull("hwchase17/react")

## Read the prompt template 
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [22]:
## Construct the ReAct agent by defining the llm, tools and prompt template
shell_Agent = create_react_agent(llm=llm,tools=tools,prompt=prompt)

# Create an agent executor by passing in the agent and tools
agent_executor = AgentExecutor(agent=shell_Agent, tools=tools, verbose=True)

In [23]:
## Run the agent executor

input = 'What are the files in the current directory?'
response = agent_executor.invoke({"input":str(input)})
response



> Entering new AgentExecutor chain...
<think>
Okay, the user is asking for the files in the current directory. I need to figure out how to get that information. Since I have access to the terminal tool, I should use a Linux command to list the files.

The standard command to list files in a directory is 'ls'. But maybe they want more details, like file types or permissions. However, the question is straightforward, just asking for the files present. The 'ls' command without options will list the names of the files and directories. 

Wait, maybe I should use 'ls -l' to get a detailed list, including permissions, size, date, etc. But the user might just want the names. However, since the question isn't specific, the safest approach is to use 'ls' to list the files. Alternatively, maybe the user wants all entries, including hidden files. But the default 'ls' doesn't show hidden files. 

Hmm, but the question is "What are the files in the current directory?" which might include both regu

/groups/ycolon/group-envs/agentic-tutorials/lib/python3.10/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


<think>
Okay, the user asked for the files in the current directory. I used the 'ls' command, which listed the files. The output shows the files are Agent_1.ipynb, chatbot_vs_agent.ipynb, dev_test, Ethanol.data, test.ipynb, and Tutorial1_Doc.md. So the answer should be these files.
</think>

Final Answer: The files in the current directory are Agent_1.ipynb, chatbot_vs_agent.ipynb, dev_test, Ethanol.data, test.ipynb, and Tutorial1_Doc.md.

> Finished chain.


{'input': 'What are the files in the current directory?',
 'output': 'The files in the current directory are Agent_1.ipynb, chatbot_vs_agent.ipynb, dev_test, Ethanol.data, test.ipynb, and Tutorial1_Doc.md.'}